In [34]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. Create embedding model

In [35]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3109.91it/s]


# 2. Create Chroma vector store

In [36]:
vector_store = Chroma(
    persist_directory="chroma_db",
    collection_name="all_documents",
    embedding_function=embedding,
)


# 3. Add documents

In [37]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"},
        id=1
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"},
        id=2
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"},
        id=3
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"},
        id=4
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"},
        id=5
    )

In [38]:
docs = [doc1, doc2, doc3, doc4, doc5]

# add documents
vector_store.add_documents(docs)

['1', '2', '3', '4', '5']

In [39]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['1', '2', '3', '4', '5'],
 'embeddings': array([[ 0.00994723,  0.06914327, -0.05147115, ..., -0.03543331,
          0.01284812,  0.01248286],
        [ 0.00127745,  0.03129854, -0.02375381, ..., -0.00518362,
         -0.03280611,  0.02737713],
        [-0.10265917,  0.02650803,  0.02271505, ..., -0.03359744,
         -0.07984944, -0.0150771 ],
        [ 0.02123396, -0.02468551, -0.04494367, ..., -0.10995807,
          0.00572556,  0.09915373],
        [ 0.01873977,  0.04382844, -0.0430425 , ..., -0.07801621,
         -0.07840683, -0.00304188]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously k

In [40]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='4', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='2', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [41]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='4', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693602323532104),
 (Document(id='2', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1493451595306396)]

In [42]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='3', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436002731323242),
 (Document(id='5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909374475479126)]

In [43]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='1', document=updated_doc1)

In [44]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['1', '2', '3', '4', '5'],
 'embeddings': array([[-0.00233744,  0.0590208 , -0.04774046, ..., -0.07264049,
          0.00276784, -0.00344091],
        [ 0.00127745,  0.03129854, -0.02375381, ..., -0.00518362,
         -0.03280611,  0.02737713],
        [-0.10265917,  0.02650803,  0.02271505, ..., -0.03359744,
         -0.07984944, -0.0150771 ],
        [ 0.02123396, -0.02468551, -0.04494367, ..., -0.10995807,
          0.00572556,  0.09915373],
        [ 0.01873977,  0.04382844, -0.0430425 , ..., -0.07801621,
         -0.07840683, -0.00304188]], shape=(5, 384)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anch

In [45]:
# delete document
vector_store.delete(ids=['1'])

In [46]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['2', '3', '4', '5'],
 'embeddings': array([[ 0.00127745,  0.03129854, -0.02375381, ..., -0.00518362,
         -0.03280611,  0.02737713],
        [-0.10265917,  0.02650803,  0.02271505, ..., -0.03359744,
         -0.07984944, -0.0150771 ],
        [ 0.02123396, -0.02468551, -0.04494367, ..., -0.10995807,
          0.00572556,  0.09915373],
        [ 0.01873977,  0.04382844, -0.0430425 , ..., -0.07801621,
         -0.07840683, -0.00304188]], shape=(4, 384)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.',
  'Ra